# 2: Prompt evaluation

This notebook showcases all the concepts from the articles under the "Prompt evaluation" section.

The course includes Python notebooks attached to the following articles: [Generating test datasets](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287739), [Model based grading](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287742), [Code based grading](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287737), and [Exercise on prompt evals](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287738).
Those notebooks were used as sources in this notebook, but the original files are not included in this repository.

In [ ]:
%pip install openai

In [ ]:
api_key = "sk-834c9dee0ded409d9f439bd9b41e1354" # Paste here for convenience. Use .env in real code for security
base_url = "https://api.deepseek.com/beta" # or "https://api.anthropic.com"
model = "deepseek-v4-flash" # or "claude-sonnet-4-0"

In [ ]:
# Generic helper functions, modified to access OpenAI-style API

from openai import OpenAI
from statistics import mean
import json

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system

    # Modified for OpenAI-style API
    if stop_sequences:
        params["stop"] = stop_sequences

    if messages[-1]["role"] == "assistant":
        params["messages"][-1]["prefix"] = True

    response = client.chat.completions.create(**params)
    return response.choices[0].message.content

A dataset generation function based on [Generating test datasets](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287739) and [Exercise on prompt evals](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287738)

In [ ]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()
print(dataset)

#with open('dataset.json', 'w') as f:
#    json.dump(dataset, f, indent=2)

Hard-coded prompt function from [Running the eval](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287743)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

Model-based grading example based on [Model based grading](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287742) and [Exercise on prompt evals](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287738)

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_test_case(test_case):
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
eval_results = run_eval(dataset)
print(eval_results)

Syntax validation based on [Code based grading](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287737)

In [ ]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

# Order of parameters is different from model grading in article code
def grade_syntax(text, test_case):
    if test_case["format"] == "json":
        return validate_json(text)
    elif test_case["format"] == "python":
        return validate_python(text)
    elif test_case["format"] == "regex":
        return validate_regex(text)
    else:
        raise ValueError(f"Invalid format: {test_case['format']}")

def combined_score(text, test_case):
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2